In [1]:
# Import libraries
import numpy as np
import pandas as pd
import pyhf
import matplotlib.pyplot as plt

# Import Bayesian analysis libraries
from bayesian_pyhf import infer
import pymc as pm
import arviz as az
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.integrate import cumulative_trapezoid
from scipy import optimize
            

In [2]:
nominal_eps = 1e-3
scaling = 0.05
fraction_outside = 1.1

# Target POT
target_pot_run1 = 2.2e20
target_pot_run3 = 5.02e20

# Masses to loop over
masses = ["0.01", "0.02", "0.03", "0.04", "0.05", "0.06", "0.07", "0.08", "0.09", "0.1", "0.2", "0.3", "0.4"]

# Meson scalings
meson_scalings = {"pi0": 11.7, "eta": 7.2}

# Correction factors
correction_dic_a = {
    "pi0": {'0.01': 11.198, '0.02': 11.145, '0.03': 10.861, '0.04': 11.543, '0.05': 10.671, 
            '0.06': 10.098, '0.07': 8.741, '0.08': 10.282, '0.09': 9.333},
    "eta": {'0.02': 10.555, '0.03': 9.759, '0.04': 7.934, '0.05': 9.387, '0.06': 9.863, 
            '0.07': 10.301, '0.08': 9.141, '0.09': 7.273, '0.1': 7.228, '0.2': 7.597, 
            '0.3': 5.219, '0.4': 4.498}
}

# Uncertainties
pot_uncert = 0.02  # 2% POT uncertainty
flux_uncert = 0.22  # 22% flux uncertainty
detvar_uncert = 0.18  # 18% detector variation uncertainty

# Cross-section uncertainties (%) 
run1_xsec_uncert_a = {
    "pi0": {"0.01": 1.16, "0.02": 2.27, "0.03": 3.48, "0.04": 4.48, "0.05": 5.50, 
            "0.06": 5.92, "0.07": 7.17, "0.08": 8.55, "0.09": 8.97},
    "eta": {"0.02": 2.09, "0.03": 3.50, "0.04": 4.33, "0.05": 6.07, "0.06": 6.26, 
            "0.07": 7.45, "0.08": 8.72, "0.09": 8.99, "0.1": 9.63, "0.2": 18.95, 
            "0.3": 22.04, "0.4": 30.34}
}

run3_xsec_uncert_a = {
    "pi0": {"0.01": 0.92, "0.02": 2.32, "0.03": 2.82, "0.04": 4.32, "0.05": 5.24, 
            "0.06": 5.93, "0.07": 6.44, "0.08": 8.94, "0.09": 7.84},
    "eta": {"0.02": 2.15, "0.03": 3.24, "0.04": 4.14, "0.05": 5.41, "0.06": 7.17, 
            "0.07": 6.41, "0.08": 7.87, "0.09": 8.32, "0.1": 9.22, "0.2": 15.78, 
            "0.3": 20.14, "0.4": 23.64}
}

# total_pot values
total_pot = {
    "run1_dt_ratio_0.6_ma_0.01_pi0": 3.583299e+20,
    "run1_dt_ratio_0.6_ma_0.02_eta": 1.971339e+22,
    "run1_dt_ratio_0.6_ma_0.02_pi0": 3.785739e+21,
    "run1_dt_ratio_0.6_ma_0.03_eta": 6.018737e+22,
    "run1_dt_ratio_0.6_ma_0.03_pi0": 1.403744e+22,
    "run1_dt_ratio_0.6_ma_0.04_eta": 2.797252e+23,
    "run1_dt_ratio_0.6_ma_0.04_pi0": 6.144923e+22,
    "run1_dt_ratio_0.6_ma_0.05_eta": 6.491310e+23,
    "run1_dt_ratio_0.6_ma_0.05_pi0": 1.594137e+23,
    "run1_dt_ratio_0.6_ma_0.06_eta": 1.333156e+24,
    "run1_dt_ratio_0.6_ma_0.06_pi0": 5.428614e+23,
    "run1_dt_ratio_0.6_ma_0.07_eta": 1.726805e+24,
    "run1_dt_ratio_0.6_ma_0.07_pi0": 2.122624e+24,
    "run1_dt_ratio_0.6_ma_0.08_eta": 3.721201e+24,
    "run1_dt_ratio_0.6_ma_0.08_pi0": 9.137349e+24,
    "run1_dt_ratio_0.6_ma_0.09_eta": 6.937703e+24,
    "run1_dt_ratio_0.6_ma_0.09_pi0": 4.752829e+25,
    "run1_dt_ratio_0.6_ma_0.1_eta": 2.632356e+25,
    "run1_dt_ratio_0.6_ma_0.2_eta": 2.642929e+27,
    "run1_dt_ratio_0.6_ma_0.3_eta": 4.548050e+29,
    "run1_dt_ratio_0.6_ma_0.4_eta": 2.856632e+32,
    "run3_dt_ratio_0.6_ma_0.01_pi0": 7.825758e+20,
    "run3_dt_ratio_0.6_ma_0.02_eta": 2.618067e+22,
    "run3_dt_ratio_0.6_ma_0.02_pi0": 3.553433e+21,
    "run3_dt_ratio_0.6_ma_0.03_eta": 1.076195e+23,
    "run3_dt_ratio_0.6_ma_0.03_pi0": 1.590684e+22,
    "run3_dt_ratio_0.6_ma_0.04_eta": 3.182032e+23,
    "run3_dt_ratio_0.6_ma_0.04_pi0": 6.326536e+22,
    "run3_dt_ratio_0.6_ma_0.05_eta": 1.458703e+24,
    "run3_dt_ratio_0.6_ma_0.05_pi0": 3.951992e+23,
    "run3_dt_ratio_0.6_ma_0.06_eta": 1.106334e+24,
    "run3_dt_ratio_0.6_ma_0.06_pi0": 8.377697e+23,
    "run3_dt_ratio_0.6_ma_0.07_eta": 3.539695e+24,
    "run3_dt_ratio_0.6_ma_0.07_pi0": 2.825310e+24,
    "run3_dt_ratio_0.6_ma_0.08_eta": 4.752001e+24,
    "run3_dt_ratio_0.6_ma_0.08_pi0": 1.088737e+25,
    "run3_dt_ratio_0.6_ma_0.09_eta": 8.286352e+24,
    "run3_dt_ratio_0.6_ma_0.09_pi0": 7.609472e+25,
    "run3_dt_ratio_0.6_ma_0.1_eta": 1.655501e+25,
    "run3_dt_ratio_0.6_ma_0.2_eta": 2.472610e+27,
    "run3_dt_ratio_0.6_ma_0.3_eta": 2.116252e+29,
    "run3_dt_ratio_0.6_ma_0.4_eta": 3.486213e+32,
}

# Background scalings Background POT
scalings_run1 = {"nu": 1.0/2.35e21, "dirt": 1.026*0.75/1.6e21, "beamoff": 0.98*(5736147/9186361.390000)}
scalings_run3 = {"nu": 1.0/1.993661e21, "dirt": 1.0*0.35/1.020e21, "beamoff": 0.98*(10385459.0/34147459.925000)}

In [3]:
# Directories
base_dir_run1 = "/home/paul/Msci/dm_sets/run1_samples/"
signal_dir_run1 = "/home/paul/Msci/dm_sets/run1_signal/"

# Background (RUN 1 ONLY)
df_nu_run1 = pd.read_csv(base_dir_run1 + "run1_nu_overlay_merged_with_weights.csv").drop_duplicates()
df_dirt_run1 = pd.read_csv(base_dir_run1 + "run1_dirt_merged_with_weights.csv").drop_duplicates()
df_offbeam_run1 = pd.read_csv(base_dir_run1 + "run1_offbeam_larcv_cropped_full_set_scores.csv").drop_duplicates()

print(f"Run 1 backgrounds - Nu: {len(df_nu_run1)}, Dirt: {len(df_dirt_run1)}, Offbeam: {len(df_offbeam_run1)}")

# Real data (RUN 1 ONLY)
data_run1 = pd.read_csv('/home/paul/Msci/dm_sets/run_data/run1_NuMI_beamon_larcv_cropped_full_set_scores.csv')

# Logit transform function
def logit_transform(score):
    return np.log(score / (1 - score))

# Luis's bins in logit space
bins = [0., 1.375, 2.75, 4.125, 5.5, 6.875]

# Filter to signal region then apply logit (RUN 1 ONLY)
df_nu_run1_sr = df_nu_run1[df_nu_run1['signal_score'] >= 0.].copy()
df_dirt_run1_sr = df_dirt_run1[df_dirt_run1['signal_score'] >= 0.].copy()
df_offbeam_run1_sr = df_offbeam_run1[df_offbeam_run1['signal_score'] >= 0.].copy()
data_run1_sr = data_run1[data_run1['signal_score'] >= 0.].copy()

# Apply logit transform
df_nu_run1_sr['logit_score'] = logit_transform(df_nu_run1_sr['signal_score'])
df_dirt_run1_sr['logit_score'] = logit_transform(df_dirt_run1_sr['signal_score'])
df_offbeam_run1_sr['logit_score'] = logit_transform(df_offbeam_run1_sr['signal_score'])
data_run1_sr['logit_score'] = logit_transform(data_run1_sr['signal_score'])

# Histogram data in logit space (RUN 1 ONLY)
hist_data_run1, _ = np.histogram(data_run1_sr['logit_score'], bins=bins)


Run 1 backgrounds - Nu: 13770, Dirt: 3331, Offbeam: 3889


In [4]:
# Background scalings (RUN 1 ONLY)
nu_scaling_run1 = target_pot_run1 * scalings_run1["nu"]
dirt_scaling_run1 = target_pot_run1 * scalings_run1["dirt"]
offbeam_scaling_run1 = scalings_run1["beamoff"]

# Background histograms in logit space (RUN 1 ONLY)
hist_nu_run1, _ = np.histogram(df_nu_run1_sr['logit_score'], bins=bins, weights=df_nu_run1_sr['weight'])
hist_dirt_run1, _ = np.histogram(df_dirt_run1_sr['logit_score'], bins=bins, weights=df_dirt_run1_sr['weight'])
hist_offbeam_run1, _ = np.histogram(df_offbeam_run1_sr['logit_score'], bins=bins)

# Apply scalings
hist_nu_run1 = hist_nu_run1 * nu_scaling_run1
hist_dirt_run1 = hist_dirt_run1 * dirt_scaling_run1
hist_offbeam_run1 = hist_offbeam_run1 * offbeam_scaling_run1

hist_bkg_run1 = hist_nu_run1 + hist_dirt_run1 + hist_offbeam_run1

# RUN 1 ONLY - no run3 concatenation
nbkg = hist_bkg_run1.tolist()

# Asimov data (background-only for fair comparison with cos(θ))
n_data = nbkg.copy()

# Background uncertainties
bkg_uncert_run1 = np.sqrt(hist_bkg_run1)
bkg_uncert_run1 = np.maximum(bkg_uncert_run1, 0.01 * hist_bkg_run1 + 0.001)
sigma_bkg = bkg_uncert_run1.tolist()

total_bkg_run1 = np.sum(hist_bkg_run1)

print(f"Background Run1: {total_bkg_run1:.2f}")
print(f"Background Total: {sum(nbkg):.2f}")
print(f"Data Total (Asimov): {sum(n_data):.2f}")


Background Run1: 75.10
Background Total: 75.10
Data Total (Asimov): 75.10


In [5]:
def get_signal_scaling(run, mass, dmode):
    """Calculate signal scaling for a given run, mass, and decay mode."""
    key = f"{run}_dt_ratio_0.6_ma_{mass}_{dmode}"
    pot = total_pot[key]
    target = target_pot_run1 if run == "run1" else target_pot_run3
    
    # Base scaling: meson_scaling * target_pot * correction term / total_pot NEED FRACTIONAL
    scale = meson_scalings[dmode] * target * correction_dic_a[dmode][mass] / pot
    return scale

In [6]:
def load_signal_for_mass(mass):
    """Load signal for a given mass (RUN 1 ONLY)."""
    
    hist_signal_run1 = np.zeros(len(bins) - 1)
    
    # Determine which decay modes to use
    if mass == "0.01":
        decay_modes = ["pi0"]
    elif float(mass) >= 0.1:
        decay_modes = ["eta"]
    else:
        decay_modes = ["pi0", "eta"]
    
    for dmode in decay_modes:
        # Run 1 ONLY
        try:
            fname_run1 = f"{signal_dir_run1}run1_dt_ratio_0.6_ma_{mass}_{dmode}_larcv_cropped_scores.csv"
            df = pd.read_csv(fname_run1).drop_duplicates()
            # Filter to signal region and apply logit
            df_sr = df[df['signal_score'] >= 0.5].copy()
            df_sr['logit_score'] = logit_transform(df_sr['signal_score'])
            scale = get_signal_scaling("run1", mass, dmode)
            hist, _ = np.histogram(df_sr['logit_score'], bins=bins)
            hist_signal_run1 += hist * scale
        except FileNotFoundError:
            print(f"  Warning: {fname_run1} not found")
    
    return hist_signal_run1  # Return only run1


In [7]:
# Store results
results_list = []

for mass in masses:
    print(f"\n{'='*60}")
    print(f"Processing mass: {mass}")
    print(f"{'='*60}")
    
    # Load signal for this mass (RUN 1 ONLY)
    hist_signal_run1 = load_signal_for_mass(mass)
    
    total_sig_run1 = np.sum(hist_signal_run1)
    
    print(f"Signal Run1: {total_sig_run1:.2f}")

    # Calculate cross-section uncertainty for this mass
    xsec_uncert_sq = 0
    if mass == "0.01":
        decay_modes = ["pi0"]
    elif float(mass) >= 0.1:
        decay_modes = ["eta"]
    else:
        decay_modes = ["pi0", "eta"]
    
    for dmode in decay_modes:
        if dmode in run1_xsec_uncert_a and mass in run1_xsec_uncert_a[dmode]:
            xsec_uncert_sq += run1_xsec_uncert_a[dmode][mass]**2
    
    xsec_uncert = np.sqrt(xsec_uncert_sq) / 100
    print(f"Cross-section uncertainty: {xsec_uncert:.1%}")
    
    # RUN 1 ONLY - no run3 concatenation
    nsig = hist_signal_run1.tolist()
    
    # Luis's factor calculation (RUN 1 ONLY)
    factor = scaling * total_bkg_run1 / total_sig_run1
    print(f"Scaling factor: {factor:.6e}")
    
    # Scale signal
    nsig_scaled = [s * factor * fraction_outside for s in nsig]
    
    # Build pyhf model with syst (RUN 1 ONLY)
    model = pyhf.Model({
        "channels": [{
            "name": "singlechannel",
            "samples": [
                {
                    "name": "signal",
                    "data": nsig_scaled,
                    "modifiers": [
                        {"name": "mu", "type": "normfactor", "data": None},
                        {"name": "pot_uncert", "type": "normsys", "data": {"hi": 1.02, "lo": 0.98}},
                        {"name": "flux_uncert", "type": "normsys", "data": {"hi": 1.22, "lo": 0.78}},
                        {"name": "det_uncert", "type": "normsys", "data": {"hi": 1.18, "lo": 0.82}},
                        {"name": "xsec_uncert", "type": "normsys", "data": {"hi": 1 + xsec_uncert, "lo": 1 - xsec_uncert}},
                    ]
                },
                {
                    "name": "background",
                    "data": nbkg,  # This is run1 only from Cell 4
                    "modifiers": [
                        {"name": "pot_uncert", "type": "normsys", "data": {"hi": 1.02, "lo": 0.98}},
                    ]
                }
            ]
        }]
    })
    
    # Frequentist
    poi_values = np.linspace(0., 10., 100)
    try:
        obs = n_data + model.config.auxdata  # n_data is Asimov (run1 bkg only)
        obs_limit, exp_limits, (scan, res) = pyhf.infer.intervals.upper_limits.upper_limit(
            obs, model, poi_values, level=0.1, return_results=True)
        
        # Convert to epsilon^2
        obs_epsilon = (nominal_eps**2) * np.sqrt(obs_limit * factor)
        exp_epsilon = (nominal_eps**2) * np.sqrt(exp_limits[2] * factor)
        
        exp_minus2 = (nominal_eps**2) * np.sqrt(exp_limits[0] * factor)
        exp_minus1 = (nominal_eps**2) * np.sqrt(exp_limits[1] * factor)
        exp_median = (nominal_eps**2) * np.sqrt(exp_limits[2] * factor)
        exp_plus1 = (nominal_eps**2) * np.sqrt(exp_limits[3] * factor)
        exp_plus2 = (nominal_eps**2) * np.sqrt(exp_limits[4] * factor)
        
        print(f"  Expected limit: μ = {exp_limits[2]:.4f}")
        print(f"  Expected limit: ε² = {exp_median:.2e}")
        print(f"  Bands: -2σ={exp_limits[0]:.3f}, -1σ={exp_limits[1]:.3f}, +1σ={exp_limits[3]:.3f}, +2σ={exp_limits[4]:.3f}")
        
        results_list.append({
            'mass': float(mass),
            'mu_exp': exp_limits[2],
            'mu_exp_minus2': exp_limits[0],
            'mu_exp_minus1': exp_limits[1],
            'mu_exp_plus1': exp_limits[3],
            'mu_exp_plus2': exp_limits[4],
            'epsilon_squared_exp': exp_median,
            'epsilon_squared_exp_minus2': exp_minus2,
            'epsilon_squared_exp_minus1': exp_minus1,
            'epsilon_squared_exp_plus1': exp_plus1,
            'epsilon_squared_exp_plus2': exp_plus2,
            'factor': factor
        })
        
    except Exception as e:
        print(f"  Frequentist error for mass {mass}: {e}")

# Create dataframe and save
df_results = pd.DataFrame(results_list)
df_results = df_results.sort_values('mass')
print("\n" + "="*60)
print("RESULTS (RUN 1 ONLY)")
print("="*60)
print(df_results)

# Save to CSV with different name
df_results.to_csv('/home/paul/Msci/plots/sensitivity/sensitivity_results_run1only_nocut.csv', index=False)
print("\nSaved to sensitivity_results_run1only.csv")



Processing mass: 0.01
Signal Run1: 253462.88
Cross-section uncertainty: 1.2%
Scaling factor: 1.481575e-05
  Expected limit: μ = 3.4218
  Expected limit: ε² = 7.12e-09
  Bands: -2σ=1.530, -1σ=2.192, +1σ=5.745, +2σ=9.772

Processing mass: 0.02
Signal Run1: 35852.16
Cross-section uncertainty: 3.1%
Scaling factor: 1.047425e-04
  Expected limit: μ = 3.3092
  Expected limit: ε² = 1.86e-08
  Bands: -2σ=1.476, -1σ=2.115, +1σ=5.570, +2σ=9.501

Processing mass: 0.03
Signal Run1: 6818.54
Cross-section uncertainty: 4.9%
Scaling factor: 5.507398e-04
  Expected limit: μ = 3.1868
  Expected limit: ε² = 4.19e-08
  Bands: -2σ=1.415, -1σ=2.033, +1σ=5.379, +2σ=9.207

Processing mass: 0.04
Signal Run1: 2029.65
Cross-section uncertainty: 6.2%
Scaling factor: 1.850197e-03
  Expected limit: μ = 3.1587
  Expected limit: ε² = 7.64e-08
  Bands: -2σ=1.401, -1σ=2.012, +1σ=5.342, +2σ=9.170

Processing mass: 0.05
Signal Run1: 595.51
Cross-section uncertainty: 8.2%
Scaling factor: 6.305918e-03
  Expected limit: μ =